# Electric cable instance segmentation
### Curated RF-DETR / COCO / PCA analysis

This notebook provides a readable analysis workflow for thin, elongated cable instances in aerial imagery.
The original `Segmentation.ipynb` remains **an experimental archive**, with all PCA/RANSAC/SAM/DeepLSD alternatives intact.

**Sequence:** setup → COCO audit → visualisation → optional training → optional inference → validation threshold → locked test evaluation → geometry diagnostics → report.

Historical README values (AP50 **0.528**, AR@10 **0.280**, angular error approximately **0.998°**) are *reported rather than recomputed*. They must not be treated as current outputs.

## 1. Configuration

Install the packages in `requirements-analysis.txt` for the evaluation path.
Training/inference require `requirements-training.txt`. Set the dataset and prediction paths in **one** configuration cell.

Expected dataset: `data/coco/{train,valid,test}/_annotations.coco.json` with images in their matching directories. No automatic Drive mount, extraction, cleanup or package uninstall is performed.

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cable_seg.core import (
    ProjectConfig, read_json, inspect_coco, instance_density,
    select_by_score, line_from_mask, axial_angle_error_deg,
)
from cable_seg.coco import (
    decode_segmentation, evaluate_coco, geometry_diagnostics, save_predictions,
)

CFG = ProjectConfig(
    dataset_root=Path("data/coco"),
    checkpoint=Path("checkpoint_best_ema.pth"),
    predictions_valid=Path("artifacts/predictions_valid.json"),
    predictions_test=Path("artifacts/predictions_test.json"),
    output_dir=Path("artifacts/reports"),
    seed=42, score_threshold=0.25,
)
random.seed(CFG.seed)
np.random.seed(CFG.seed)
print(CFG)

## 2. Read-only COCO audit

Inspect image/annotation counts, category consistency, empty images, density and invalid image references. This notebook **never modifies** source annotation JSON files.

In [ ]:
split_data = {}
summaries = []
for split in ("train", "valid", "test"):
    path = CFG.ann_path(split)
    if not path.exists():
        print(f"{split}: no annotation file at {path}")
        continue
    data = read_json(path)
    split_data[split] = data
    summaries.append({"split": split, **inspect_coco(data)})
if summaries:
    display(pd.DataFrame(summaries).set_index("split"))
    if any(s["orphan_annotations"] for s in summaries):
        raise ValueError("COCO annotations refer to missing image IDs")
else:
    print("No dataset found. Configure CFG.dataset_root.")

### 2.1 Cable-instance density

Density is relevant for parallel cables and occluded scenes. The plot describes annotations; it is not a model-performance result.

In [ ]:
if split_data:
    fig, axes = plt.subplots(1, len(split_data),
                             figsize=(5*len(split_data), 3.5), squeeze=False)
    for ax, (name, data) in zip(axes.flat, split_data.items()):
        arr = list(instance_density(data).values())
        ax.hist(arr, bins=min(30, max(1, len(set(arr)))), edgecolor="white")
        ax.set(title=f"{name}: cable instances", xlabel="GT cables per image",
               ylabel="Image count")
    plt.tight_layout()
    plt.show()

### 2.2 Sample annotation

Overlay the union of cable masks on one annotated sample image. Individual COCO annotations remain distinct for scoring.

In [ ]:
from PIL import Image

sample_split = "valid" if "valid" in split_data else (
    next(iter(split_data)) if split_data else None)
if sample_split:
    data = split_data[sample_split]
    sample = next((im for im in data["images"]
                   if any(a["image_id"] == im["id"] for a in data["annotations"])), None)
    if sample:
        path = CFG.dataset_root / sample_split / sample["file_name"]
        if path.exists():
            image = np.asarray(Image.open(path).convert("RGB"))
            selected = [a for a in data["annotations"] if a["image_id"] == sample["id"]]
            combined = np.zeros(image.shape[:2], dtype=bool)
            for annotation in selected:
                combined |= decode_segmentation(
                    annotation["segmentation"],
                    height=sample["height"], width=sample["width"])
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            axes[0].imshow(image)
            axes[0].set_title("Input")
            axes[1].imshow(image)
            axes[1].imshow(np.ma.masked_where(~combined, combined),
                           alpha=0.4, cmap="autumn")
            axes[1].set_title(f"GT cable pixels ({len(selected)} instances)")
            for ax in axes: ax.axis("off")
            plt.tight_layout()
            plt.show()
        else:
            print("Image missing:", path)

## 3. Optional RF-DETR training (off by default)

The original research used **multiple distinct** training recipes; a representative 60-epoch configuration is shown here for orientation. Its architecture/API must be verified against the installed RF-DETR version and the historical weights.

Train using the train/valid splits only. **Do not tune hyperparameters on test.** This expensive step is deliberately disabled unless explicitly enabled.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from rfdetr import RFDETRSegPreview
    if not CFG.ann_path("train").exists() or not CFG.ann_path("valid").exists():
        raise FileNotFoundError("train and valid COCO annotations required")
    model = RFDETRSegPreview()
    model.train(
        dataset_dir=str(CFG.dataset_root), dataset_file="roboflow",
        epochs=60, batch_size=1, grad_accum_steps=8,
        lr=1e-4, lr_encoder=1e-5,
        resolution=1200, clip_max_norm=0.1, use_ema=True,
        output_dir="artifacts/train_run",
    )
else:
    print("Training skipped (default).")

## 4. Optional inference (off by default)

Use existing COCO predictions to avoid rerunning the GPU. The optional adapter uses the public `model.predict()` API and converts masks to COCO RLE. Verify class-ID mapping, RF-DETR architecture, and checkpoint version. A Git LFS pointer cannot act as a checkpoint.

Infer at a low cutoff (0.05); optimize confidence **only on validation**.

In [ ]:
RUN_INFERENCE = False
MODEL_CLASS = "RFDETRSegPreview"  # verify model/checkpoint compatibility
if RUN_INFERENCE:
    import rfdetr
    from cable_seg.inference import predict_coco
    if not CFG.checkpoint.is_file() or CFG.checkpoint.stat().st_size < 1024:
        raise FileNotFoundError("Fetch the full LFS checkpoint first")
    model = getattr(rfdetr, MODEL_CLASS)(pretrain_weights=str(CFG.checkpoint))
    for split, output in (
        ("valid", CFG.predictions_valid), ("test", CFG.predictions_test)
    ):
        if split not in split_data:
            raise FileNotFoundError(f"No {split} COCO annotations")
        records = predict_coco(model, split_data[split],
                               CFG.dataset_root / split, threshold=0.05)
        save_predictions(output, records)
        print(f"{split}: {len(records)} detections -> {output}")
else:
    print("Inference skipped: expecting precomputed prediction JSONs.")

## 5. Choose score threshold on validation

A test-based threshold sweep creates selection bias. Choose the highest validation AP50 among predeclared cutoffs, with a lower-cutoff tie-break. The selected cutoff is **frozen** before test.

COCO recall metric indices for `maxDets=[1,10,100]`: `stats[6]` is **AR@1**, `stats[7]` is **AR@10**, `stats[8]` is **AR@100**. At least one historical cell incorrectly labels `stats[6]`.

In [ ]:
validation_predictions = (
    read_json(CFG.predictions_valid) if CFG.predictions_valid.exists() else None
)
selected_threshold = CFG.score_threshold
validation_metrics = None
if validation_predictions is not None and CFG.ann_path("valid").exists():
    sweep = []
    for candidate in (0.05, 0.10, 0.20, 0.30, 0.40, 0.50):
        filtered = select_by_score(validation_predictions, candidate)
        sweep.append({"threshold": candidate,
                      **evaluate_coco(CFG.ann_path("valid"), filtered)})
    display(pd.DataFrame(sweep).round(4))
    validation_metrics = max(sweep, key=lambda row: (row["AP50"], -row["threshold"]))
    selected_threshold = validation_metrics["threshold"]
    print(f"Frozen score threshold: {selected_threshold:.2f}")
else:
    print(f"Validation predictions not found. Fixed score threshold: {selected_threshold:.2f}")

## 6. Locked test COCO metrics

Evaluate the test split only after threshold selection. These are official COCO segmentation scores; they cannot be compared to the historical notebook until dataset, checkpoint and preprocessing versions are reconciled.

In [ ]:
test_predictions = (
    read_json(CFG.predictions_test) if CFG.predictions_test.exists() else None
)
test_filtered = None
if test_predictions is not None and CFG.ann_path("test").exists():
    test_filtered = select_by_score(test_predictions, selected_threshold)
    test_metrics = evaluate_coco(CFG.ann_path("test"), test_filtered)
    display(pd.DataFrame([test_metrics], index=["test"]).round(4))
else:
    print("No test predictions/annotations. No COCO metric was computed.")

## 7. Geometry-aware PCA diagnostics

For each greedily **one-to-one IoU-matched** GT/prediction pair (mask IoU ≥ 0.5), fit an undirected line through the mask pixels with PCA and compute axial error modulo 180°. Report matched-pair coverage to avoid hiding hard instances.

This is a diagnostic and **not** COCO matching or a reproduction of the historical ~0.998° angular error.

In [ ]:
if test_filtered is not None:
    geometry = geometry_diagnostics(
        CFG.ann_path("test"), test_filtered,
        min_iou=0.5, min_line_pixels=CFG.min_line_pixels)
    display(pd.DataFrame([geometry], index=["test"]).round(4))
else:
    print("Geometry diagnostics require test predictions.")

## 8. Save report and provenance

Persist parameters and newly computed metrics only when evaluation actually ran. No invented metrics or rerun claims are emitted.

In [ ]:
if test_filtered is not None:
    CFG.output_dir.mkdir(parents=True, exist_ok=True)
    report = {
        "dataset_root": str(CFG.dataset_root),
        "test_ground_truth": str(CFG.ann_path("test")),
        "valid_predictions": str(CFG.predictions_valid),
        "test_predictions": str(CFG.predictions_test),
        "seed": CFG.seed,
        "selected_threshold": selected_threshold,
        "selected_on_validation": validation_metrics is not None,
        "validation_metrics": validation_metrics,
        "test_coco_metrics": test_metrics,
        "geometry_diagnostics": geometry,
        "note": "Newly evaluated metrics; historical values not independently reproduced",
    }
    target = CFG.output_dir / "evaluation_summary.json"
    target.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Report saved:", target)
else:
    print("No report generated without test evaluation.")

## 9. Discussion and future evaluation

Thin, filamentary cable masks may have low pixel overlap despite plausible global orientation. This motivates measuring **both** segmentation quality and geometric consistency.

**Known limits:** dense scenes, touching/occluded cables, broken masks, variations in cable thickness, and sensitivity to matching and instance count.

**Next rigorous experiments:** freeze the dataset split and model checkpoint, keep preprocessing constant, test AP/AR stratified by scene density and compare PCA/RANSAC/SAM/DeepLSD only through matching validation/test protocols.

**Research integrity:** this curated workflow does not claim new accuracy improvements or rerun the archived 106 experimental cells. The original notebook remains the detailed experimental record.